****
**Feature Engineering**
****

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

**Setup and Load Data**
****

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
Base_Path = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/'
Clean     = Base_Path + 'cleaned/'
FE_Path   = Base_Path + 'feature_engineered/'
os.makedirs(FE_Path, exist_ok=True)

In [ ]:
# Load cleaned data
print("Loading cleaned data...")
train = pd.read_csv(Clean + 'train_cleaned.csv')
test  = pd.read_csv(Clean + 'test_cleaned.csv')

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")

Loading cleaned data...


**Verify Cleaned Data**
****

In [ ]:
print("========== LOADED DATA VERIFICATION ==========")
print(f"Nulls in train        : {train.isnull().sum().sum()}")
print(f"Nulls in test         : {test.isnull().sum().sum()}")
print(f"Fraud ratio           : {train['isFraud'].mean():.4f}")
print(f"isFraud not in test   : {'isFraud' not in test.columns}")

========== LOADED DATA VERIFICATION ==========
Nulls in train        : 0
Nulls in test         : 0
Fraud ratio           : 0.0350
isFraud not in test   : True


In [ ]:
# Check key columns exist
key_cols = ['TransactionID', 'TransactionDT', 'TransactionAmt',
            'TransactionAmt_log', 'DeviceType', 'DeviceInfo',
            'card1', 'card2', 'addr1', 'P_emaildomain', 'R_emaildomain']

print(f"\nKey columns check:")
for col in key_cols:
    status = 'OK' if col in train.columns else 'MISSING'
    print(f"  {col:<25} : {status}")


Key columns check:
  TransactionID             : OK
  TransactionDT             : OK
  TransactionAmt            : OK
  TransactionAmt_log        : OK
  DeviceType                : OK
  DeviceInfo                : OK
  card1                     : OK
  card2                     : OK
  addr1                     : OK
  P_emaildomain             : OK
  R_emaildomain             : OK


In [ ]:
# Check dtypes summary
print(f"\nDtype summary:")
print(train.dtypes.value_counts())
print("===============================================")


Dtype summary:
float64    390
object      29
int64       11
Name: count, dtype: int64


STEP 1: Time-Based Train / Validation Split
============================================================

In [ ]:
# TransactionDT is seconds from a reference point, not a real timestamp
# But it is chronologically ordered, so we split by value not randomly

print("TransactionDT distribution:")
print(f"  Min  : {train['TransactionDT'].min()}")
print(f"  Max  : {train['TransactionDT'].max()}")
print(f"  Mean : {train['TransactionDT'].mean():.0f}")

TransactionDT distribution:
  Min  : 86400
  Max  : 15811131
  Mean : 7372311


In [ ]:
# Visualise the split point
split_percentile = 80
split_value = np.percentile(train['TransactionDT'], split_percentile)
print(f"\nSplit at 80th percentile : {split_value:.0f}")

# Apply split
train_df = train[train['TransactionDT'] <= split_value].copy()
val_df   = train[train['TransactionDT'] >  split_value].copy()

print(f"\n========== SPLIT SUMMARY ==========")
print(f"Full train shape  : {train.shape}")
print(f"Train set shape   : {train_df.shape}")
print(f"Val set shape     : {val_df.shape}")
print(f"Train fraud ratio : {train_df['isFraud'].mean():.4f}")
print(f"Val fraud ratio   : {val_df['isFraud'].mean():.4f}")
print(f"Train % of data   : {len(train_df)/len(train)*100:.1f}%")
print(f"Val   % of data   : {len(val_df)/len(train)*100:.1f}%")
print("====================================")


Split at 80th percentile : 12192854

========== SPLIT SUMMARY ==========
Full train shape  : (590540, 430)
Train set shape   : (472432, 430)
Val set shape     : (118108, 430)
Train fraud ratio : 0.0351
Val fraud ratio   : 0.0344
Train % of data   : 80.0%
Val   % of data   : 20.0%


In [ ]:
# Verify no time leakage
print(f"\nLeakage check:")
print(f"  Max TransactionDT in train : {train_df['TransactionDT'].max():.0f}")
print(f"  Min TransactionDT in val   : {val_df['TransactionDT'].min():.0f}")
print(f"  Overlap                    : {train_df['TransactionDT'].max() >= val_df['TransactionDT'].min()}")


Leakage check:
  Max TransactionDT in train : 12192842
  Min TransactionDT in val   : 12192900
  Overlap                    : False


STEP 2: V Column Reduction
============================================================

In [ ]:
v_cols = [c for c in train_df.columns if c.startswith('V')]
print(f"V columns before reduction : {len(v_cols)}")

V columns before reduction : 339


In [ ]:
# ------------------------------------------------------------
# STEP 2a: Remove Near-Zero Variance V Columns
# Columns where almost every value is -999 (was missing)
# carry no signal, just noise
# ------------------------------------------------------------
from sklearn.feature_selection import VarianceThreshold

# Compute variance on training set only
v_data_train = train_df[v_cols].copy()

# Threshold: if >95% of values are identical, drop it
variance_selector = VarianceThreshold(threshold=0.05)
variance_selector.fit(v_data_train)

# Get columns that passed variance threshold
v_high_variance = [v_cols[i] for i, supported
                   in enumerate(variance_selector.get_support())
                   if supported]

v_low_variance  = [c for c in v_cols if c not in v_high_variance]

print(f"\n[Variance Filter]")
print(f"  V cols with sufficient variance : {len(v_high_variance)}")
print(f"  V cols dropped (near-zero var)  : {len(v_low_variance)}")
print(f"  Dropped cols                    : {sorted(v_low_variance)}")


[Variance Filter]
  V cols with sufficient variance : 339
  V cols dropped (near-zero var)  : 0
  Dropped cols                    : []


In [ ]:
# ------------------------------------------------------------
# STEP 2b: Correlation Filter on Remaining V Columns
# Among highly correlated pairs (>0.95), keep only one
# ------------------------------------------------------------
print(f"\n[Correlation Filter]")
print(f"Running correlation matrix on {len(v_high_variance)} remaining V cols...")

v_data_filtered = train_df[v_high_variance].copy()
corr_matrix     = v_data_filtered.corr().abs()

# Upper triangle only to avoid duplicate pairs
upper_triangle  = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)


[Correlation Filter]
Running correlation matrix on 339 remaining V cols...


In [ ]:
# Find columns where ANY correlation with another column exceeds threshold
corr_threshold  = 0.95
cols_to_drop_corr = [col for col in upper_triangle.columns
                     if any(upper_triangle[col] > corr_threshold)]

v_final = [c for c in v_high_variance if c not in cols_to_drop_corr]

print(f"  Correlation threshold           : {corr_threshold}")
print(f"  V cols dropped (high corr)      : {len(cols_to_drop_corr)}")
print(f"  V cols remaining after filter   : {len(v_final)}")

  Correlation threshold           : 0.95
  V cols dropped (high corr)      : 279
  V cols remaining after filter   : 60


In [ ]:
# ------------------------------------------------------------
# STEP 2c: Apply reduction to all datasets
# ------------------------------------------------------------
all_v_to_drop = v_low_variance + cols_to_drop_corr

train_df.drop(columns=all_v_to_drop, inplace=True)
val_df.drop(columns=all_v_to_drop,   inplace=True)
test.drop(columns=[c for c in all_v_to_drop if c in test.columns], inplace=True)

print(f"\n========== V REDUCTION SUMMARY ==========")
print(f"V cols before        : {len(v_cols)}")
print(f"Dropped low variance : {len(v_low_variance)}")
print(f"Dropped high corr    : {len(cols_to_drop_corr)}")
print(f"V cols remaining     : {len(v_final)}")
print(f"Train shape now      : {train_df.shape}")
print(f"Val shape now        : {val_df.shape}")
print(f"Test shape now       : {test.shape}")
print(f"==========================================")


========== V REDUCTION SUMMARY ==========
V cols before        : 339
Dropped low variance : 0
Dropped high corr    : 279
V cols remaining     : 60
Train shape now      : (472432, 151)
Val shape now        : (118108, 151)
Test shape now       : (506691, 150)


In [ ]:
# ------------------------------------------------------------
# STEP 2d: Verify fraud ratio preserved after column drop
# ------------------------------------------------------------
print(f"\nFraud ratio check after V reduction:")
print(f"  Train : {train_df['isFraud'].mean():.4f}")
print(f"  Val   : {val_df['isFraud'].mean():.4f}")


Fraud ratio check after V reduction:
  Train : 0.0351
  Val   : 0.0344


STEP 3: UID Engineering
============================================================

In [ ]:
print("Creating UID features...")

# ------------------------------------------------------------
# STEP 3a: Standardize component columns to string first
# Ensures consistent concatenation across all datasets
# ------------------------------------------------------------
uid_component_cols = ['card1', 'card2', 'card3', 'card5',
                      'addr1', 'addr2',
                      'P_emaildomain']

for col in uid_component_cols:
    for df in [train_df, val_df, test]:
        df[col] = df[col].astype(str).str.strip().str.lower()

print(f"Standardized {len(uid_component_cols)} UID component columns")

Creating UID features...
Standardized 7 UID component columns


In [ ]:
# ------------------------------------------------------------
# STEP 3b: Build UID combinations
# Each UID captures a different level of user identity
# More components = more specific identity fingerprint
# ------------------------------------------------------------

# UID1: Card identity only (broadest)
# Catches same card used across transactions
for df in [train_df, val_df, test]:
    df['UID1'] = (df['card1'].astype(str) + '_' +
                  df['card2'].astype(str))

# UID2: Card + billing address
# Narrows to card used from same location
for df in [train_df, val_df, test]:
    df['UID2'] = (df['card1'].astype(str) + '_' +
                  df['card2'].astype(str) + '_' +
                  df['addr1'].astype(str))

# UID3: Card + address + email (most specific)
# Approximates a unique customer account
for df in [train_df, val_df, test]:
    df['UID3'] = (df['card1'].astype(str) + '_' +
                  df['card2'].astype(str) + '_' +
                  df['addr1'].astype(str) + '_' +
                  df['P_emaildomain'].astype(str))

# UID4: Card + address + both addresses
# Captures billing/shipping combination
for df in [train_df, val_df, test]:
    df['UID4'] = (df['card1'].astype(str) + '_' +
                  df['addr1'].astype(str) + '_' +
                  df['addr2'].astype(str))

print(f"Created 4 UID columns: UID1, UID2, UID3, UID4")

Created 4 UID columns: UID1, UID2, UID3, UID4


In [ ]:
# ------------------------------------------------------------
# STEP 3c: Verify UID uniqueness and coverage
# More unique values = more granular identity separation
# ------------------------------------------------------------
print(f"\n========== UID SUMMARY ==========")
for uid in ['UID1', 'UID2', 'UID3', 'UID4']:
    n_unique_train = train_df[uid].nunique()
    n_unique_val   = val_df[uid].nunique()

    # Check how many UIDs in val were seen in train
    # Unseen UIDs = new users = higher risk signal
    train_uid_set  = set(train_df[uid].unique())
    val_uid_set    = set(val_df[uid].unique())
    unseen_in_val  = len(val_uid_set - train_uid_set)
    unseen_pct     = unseen_in_val / len(val_uid_set) * 100

    print(f"\n  {uid}:")
    print(f"    Unique in train       : {n_unique_train:,}")
    print(f"    Unique in val         : {n_unique_val:,}")
    print(f"    Unseen UIDs in val    : {unseen_in_val:,} ({unseen_pct:.1f}%)")


========== UID SUMMARY ==========

  UID1:
    Unique in train       : 13,458
    Unique in val         : 7,452
    Unseen UIDs in val    : 1,062 (14.3%)

  UID2:
    Unique in train       : 37,551
    Unique in val         : 16,745
    Unseen UIDs in val    : 3,764 (22.5%)

  UID3:
    Unique in train       : 82,218
    Unique in val         : 31,618
    Unseen UIDs in val    : 9,962 (31.5%)

  UID4:
    Unique in train       : 36,338
    Unique in val         : 16,271
    Unseen UIDs in val    : 3,348 (20.6%)


In [ ]:
# ------------------------------------------------------------
# STEP 3d: Fraud rate per UID3 (most specific)
# Quick sanity check - do some UIDs have higher fraud rates?
# ------------------------------------------------------------
uid3_fraud = train_df.groupby('UID3')['isFraud'].agg(['mean', 'count'])
uid3_fraud.columns = ['fraud_rate', 'transaction_count']
uid3_fraud = uid3_fraud[uid3_fraud['transaction_count'] >= 5]  # min 5 transactions

print(f"\n  UID3 fraud rate distribution:")
print(f"    UIDs with >=5 transactions  : {len(uid3_fraud):,}")
print(f"    Mean fraud rate across UIDs : {uid3_fraud['fraud_rate'].mean():.4f}")
print(f"    UIDs with 100% fraud rate   : {(uid3_fraud['fraud_rate'] == 1.0).sum():,}")
print(f"    UIDs with 0% fraud rate     : {(uid3_fraud['fraud_rate'] == 0.0).sum():,}")
print(f"    Top 5 highest fraud UIDs:")
print(uid3_fraud.nlargest(5, 'fraud_rate')[['fraud_rate', 'transaction_count']])
print(f"==================================")


  UID3 fraud rate distribution:
    UIDs with >=5 transactions  : 17,466
    Mean fraud rate across UIDs : 0.0299
    UIDs with 100% fraud rate   : 103
    UIDs with 0% fraud rate     : 15,102
    Top 5 highest fraud UIDs:
                                  fraud_rate  transaction_count
UID3                                                           
10057_225.0_269.0_bellsouth.net          1.0                  7
10486_514.0_220.0_icloud.com             1.0                  7
10859_231.0_272.0_hotmail.com            1.0                  5
10960_567.0_204.0_embarqmail.com         1.0                  7
11031_555.0_327.0_gmail.com              1.0                  5


In [ ]:
# ------------------------------------------------------------
# STEP 3e: Shape check
# ------------------------------------------------------------
print(f"\nShape after UID engineering:")
print(f"  Train : {train_df.shape}")
print(f"  Val   : {val_df.shape}")
print(f"  Test  : {test.shape}")


Shape after UID engineering:
  Train : (472432, 155)
  Val   : (118108, 155)
  Test  : (506691, 154)


STEP 4: Device Fingerprint Extraction
============================================================

In [ ]:
print("Sample DeviceInfo values:")
print(train_df['DeviceInfo'].value_counts().head(20))
print(f"\nTotal unique DeviceInfo values: {train_df['DeviceInfo'].nunique()}")

Sample DeviceInfo values:
DeviceInfo
unknown                        372896
windows                         39681
ios device                      17083
macos                           10843
trident/7.0                      6453
rv:11.0                          1651
rv:57.0                           957
sm-j700m build/mmb29k             441
sm-g610m build/mmb29k             385
sm-g531h build/lmy48b             341
sm-g955u build/nrd90m             313
sm-g950u build/nrd90m             284
sm-g935f build/nrd90m             273
rv:58.0                           262
ale-l23 build/huaweiale-l23       258
sm-g930v build/nrd90m             250
rv:52.0                           236
samsung                           210
sm-g532m build/mmb29t             202
sm-n950u build/nmf26x             201
Name: count, dtype: int64

Total unique DeviceInfo values: 1633


In [ ]:
# ------------------------------------------------------------
# STEP 4a: Extract Device Brand
# ------------------------------------------------------------
def extract_device_brand(device_str):
    device_str = str(device_str).lower().strip()

    if device_str == 'unknown' or device_str == 'nan':
        return 'unknown'

    # Windows devices
    if 'windows' in device_str:
        return 'windows'

    # Apple devices
    if any(x in device_str for x in ['ios', 'macos', 'mac os',
                                       'iphone', 'ipad']):
        return 'apple'

    # Samsung
    if 'samsung' in device_str or 'sm-' in device_str:
        return 'samsung'

    # Huawei
    if 'huawei' in device_str or 'ale-' in device_str:
        return 'huawei'

    # LG
    if device_str.startswith('lg') or 'lg-' in device_str:
        return 'lg'

    # Motorola
    if any(x in device_str for x in ['motorola', 'moto ', 'xt']):
        return 'motorola'

    # Linux
    if 'linux' in device_str:
        return 'linux'

    # Android generic
    if 'android' in device_str:
        return 'android_other'

    # Trident/IE based browsers
    if 'trident' in device_str:
        return 'ie_browser'

    return 'other'

# Apply to all datasets
for df in [train_df, val_df, test]:
    df['device_brand'] = df['DeviceInfo'].apply(extract_device_brand)

print(f"\nDevice brand distribution in train:")
brand_dist = train_df.groupby('device_brand').agg(
    count        = ('device_brand', 'count'),
    fraud_rate   = ('isFraud', 'mean')
).sort_values('count', ascending=False)
print(brand_dist)


Device brand distribution in train:
                count  fraud_rate
device_brand                     
unknown        372896    0.025988
windows         39722    0.062132
apple           27927    0.044760
samsung          9785    0.115074
other            9020    0.129047
ie_browser       6453    0.011468
motorola         2519    0.159984
lg               2111    0.112743
huawei           1843    0.098752
linux              94    0.000000
android_other      62    0.048387


In [ ]:
# ------------------------------------------------------------
# STEP 4b: Extract Device Type Consistency Flag
# DeviceType should align with DeviceInfo brand
# Mismatch can indicate spoofed device information
# ------------------------------------------------------------
def is_device_consistent(row):
    brand      = str(row['device_brand']).lower()
    devicetype = str(row['DeviceType']).lower()

    # Mobile brands should have mobile DeviceType
    mobile_brands = ['samsung', 'apple', 'huawei',
                     'lg', 'motorola', 'android_other']
    desktop_brands = ['windows', 'linux', 'macos', 'ie_browser']

    if brand in mobile_brands and devicetype == 'mobile':
        return 1   # Consistent
    if brand in desktop_brands and devicetype == 'desktop':
        return 1   # Consistent
    if brand == 'unknown':
        return 0   # Cannot determine

    return -1      # Inconsistent - potential spoofing signal

for df in [train_df, val_df, test]:
    df['device_consistent'] = df.apply(is_device_consistent, axis=1)

print(f"\nDevice consistency distribution:")
consistency = train_df.groupby('device_consistent').agg(
    count      = ('device_consistent', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(consistency)
print("\nNote: -1 = inconsistent (spoofing signal), 0 = unknown, 1 = consistent")


Device consistency distribution:
                    count  fraud_rate
device_consistent                    
-1                  20915    0.077695
 0                 372896    0.025988
 1                  78621    0.067196

Note: -1 = inconsistent (spoofing signal), 0 = unknown, 1 = consistent


In [ ]:
# ------------------------------------------------------------
# STEP 4c: Device Risk Flag
# Combine brand and device type into a single risk signal
# ------------------------------------------------------------
for df in [train_df, val_df, test]:
    df['device_known'] = (df['device_brand'] != 'unknown').astype(np.int8)

print(f"\nDevice known rate     : {train_df['device_known'].mean():.4f}")
print(f"Fraud rate known      : {train_df[train_df['device_known']==1]['isFraud'].mean():.4f}")
print(f"Fraud rate unknown    : {train_df[train_df['device_known']==0]['isFraud'].mean():.4f}")


Device known rate     : 0.2107
Fraud rate known      : 0.0694
Fraud rate unknown    : 0.0260


In [ ]:
# ------------------------------------------------------------
# STEP 4d: Summary
# ------------------------------------------------------------
new_device_cols = ['device_brand', 'device_consistent', 'device_known']

print(f"\n========== DEVICE FINGERPRINT SUMMARY ==========")
print(f"New features created  : {new_device_cols}")
print(f"Train shape           : {train_df.shape}")
print(f"Val shape             : {val_df.shape}")
print(f"Test shape            : {test.shape}")
print(f"=================================================")


========== DEVICE FINGERPRINT SUMMARY ==========
New features created  : ['device_brand', 'device_consistent', 'device_known']
Train shape           : (472432, 158)
Val shape             : (118108, 158)
Test shape            : (506691, 157)


STEP 5: Email Domain Features
============================================================

In [ ]:
print("P_emaildomain distribution:")
print(train_df['P_emaildomain'].value_counts().head(15))
print(f"\nUnique P_emaildomain : {train_df['P_emaildomain'].nunique()}")
print(f"Unique R_emaildomain : {train_df['R_emaildomain'].nunique()}")

P_emaildomain distribution:
P_emaildomain
gmail.com        182692
yahoo.com         80854
unknown           73586
hotmail.com       37158
anonymous.com     29961
aol.com           22618
comcast.net        6496
icloud.com         5031
outlook.com        4080
msn.com            3340
att.net            3133
live.com           2475
sbcglobal.net      2395
verizon.net        2252
ymail.com          1914
Name: count, dtype: int64

Unique P_emaildomain : 60
Unique R_emaildomain : 61


In [ ]:
# ------------------------------------------------------------
# STEP 5a: Email Domain Risk Grouping
# Group domains by risk level based on fraud rate in training
# ------------------------------------------------------------

# Compute fraud rate per P_emaildomain on training set ONLY
p_email_fraud = train_df.groupby('P_emaildomain')['isFraud'].agg(
    ['mean', 'count']
).reset_index()
p_email_fraud.columns = ['P_emaildomain', 'p_email_fraud_rate', 'p_email_count']

print(f"\nTop 15 P_emaildomain by fraud rate (min 100 transactions):")
print(p_email_fraud[p_email_fraud['p_email_count'] >= 100]
      .sort_values('p_email_fraud_rate', ascending=False)
      .head(15))


Top 15 P_emaildomain by fraud rate (min 100 transactions):
      P_emaildomain  p_email_fraud_rate  p_email_count
29         mail.com            0.194690            452
0           aim.com            0.149254            268
36       outlook.es            0.129973            377
35      outlook.com            0.091667           4080
21       hotmail.es            0.066946            239
19      hotmail.com            0.052021          37158
16        gmail.com            0.044003         182692
46   suddenlink.net            0.035088            114
12   embarqmail.com            0.033333            210
14  frontiernet.net            0.032680            153
25         live.com            0.031919           2475
23       icloud.com            0.031803           5031
9       comcast.net            0.030018           6496
48          unknown            0.029829          73586
26      live.com.mx            0.028846            624


In [ ]:
# ------------------------------------------------------------
# STEP 5b: Domain Risk Category
# Based on observed fraud rates in training data
# Low    : < 2% fraud rate (mainstream trusted domains)
# Medium : 2-8% fraud rate
# High   : > 8% fraud rate
# Unknown: domain not seen or missing
# ------------------------------------------------------------
def categorize_email_risk(domain, fraud_rate_map, count_map,
                           min_count=50):
    domain = str(domain).lower().strip()

    if domain == 'unknown' or domain == 'nan':
        return 'no_email'

    if domain not in fraud_rate_map:
        return 'rare_domain'

    if count_map[domain] < min_count:
        return 'rare_domain'

    rate = fraud_rate_map[domain]

    if rate < 0.02:
        return 'low_risk'
    elif rate < 0.08:
        return 'medium_risk'
    else:
        return 'high_risk'

# Build lookup maps from training data only
p_fraud_rate_map = dict(zip(p_email_fraud['P_emaildomain'],
                             p_email_fraud['p_email_fraud_rate']))
p_count_map      = dict(zip(p_email_fraud['P_emaildomain'],
                             p_email_fraud['p_email_count']))

# Apply to all datasets
for df in [train_df, val_df, test]:
    df['P_email_risk'] = df['P_emaildomain'].apply(
        lambda x: categorize_email_risk(x, p_fraud_rate_map, p_count_map)
    )

print(f"\nP_email_risk distribution in train:")
email_risk = train_df.groupby('P_email_risk').agg(
    count      = ('P_email_risk', 'count'),
    fraud_rate = ('isFraud', 'mean')
).sort_values('fraud_rate', ascending=False)
print(email_risk)


P_email_risk distribution in train:
               count  fraud_rate
P_email_risk                    
high_risk       5244    0.110984
medium_risk   374778    0.036427
no_email       73586    0.029829
low_risk       18656    0.009112
rare_domain      168    0.000000


In [ ]:
# ------------------------------------------------------------
# STEP 5c: Repeat for R_emaildomain
# ------------------------------------------------------------
r_email_fraud = train_df.groupby('R_emaildomain')['isFraud'].agg(
    ['mean', 'count']
).reset_index()
r_email_fraud.columns = ['R_emaildomain', 'r_email_fraud_rate', 'r_email_count']

r_fraud_rate_map = dict(zip(r_email_fraud['R_emaildomain'],
                             r_email_fraud['r_email_fraud_rate']))
r_count_map      = dict(zip(r_email_fraud['R_emaildomain'],
                             r_email_fraud['r_email_count']))

for df in [train_df, val_df, test]:
    df['R_email_risk'] = df['R_emaildomain'].apply(
        lambda x: categorize_email_risk(x, r_fraud_rate_map, r_count_map)
    )

print(f"\nR_email_risk distribution in train:")
r_email_risk = train_df.groupby('R_email_risk').agg(
    count      = ('R_email_risk', 'count'),
    fraud_rate = ('isFraud', 'mean')
).sort_values('fraud_rate', ascending=False)
print(r_email_risk)


R_email_risk distribution in train:
               count  fraud_rate
R_email_risk                    
high_risk      51014    0.117517
rare_domain      569    0.072056
medium_risk    54343    0.053898
no_email      358812    0.021142
low_risk        7694    0.006239


In [ ]:
# ------------------------------------------------------------
# STEP 5d: P vs R Email Match Flag
# Same purchaser and recipient email = likely same person
# Different emails = could indicate card-not-present fraud
# ------------------------------------------------------------
for df in [train_df, val_df, test]:
    df['email_match'] = (
        df['P_emaildomain'] == df['R_emaildomain']
    ).astype(np.int8)

    # Both emails missing
    df['both_email_missing'] = (
        (df['P_emaildomain'] == 'unknown') &
        (df['R_emaildomain'] == 'unknown')
    ).astype(np.int8)

    # Only one email missing
    df['one_email_missing'] = (
        (df['P_emaildomain'] == 'unknown') ^
        (df['R_emaildomain'] == 'unknown')
    ).astype(np.int8)

print(f"\nEmail match fraud analysis:")
email_match = train_df.groupby('email_match').agg(
    count      = ('email_match', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(email_match)

print(f"\nBoth email missing fraud rate : "
      f"{train_df[train_df['both_email_missing']==1]['isFraud'].mean():.4f}")
print(f"One email missing fraud rate  : "
      f"{train_df[train_df['one_email_missing']==1]['isFraud'].mean():.4f}")
print(f"No email missing fraud rate   : "
      f"{train_df[(train_df['both_email_missing']==0) & (train_df['one_email_missing']==0)]['isFraud'].mean():.4f}")


Email match fraud analysis:
              count  fraud_rate
email_match                    
0            320649    0.021531
1            151783    0.063874

Both email missing fraud rate : 0.0261
One email missing fraud rate  : 0.0211
No email missing fraud rate   : 0.0806


In [ ]:
# ------------------------------------------------------------
# STEP 5e: Summary
# ------------------------------------------------------------
new_email_cols = ['P_email_risk', 'R_email_risk',
                  'email_match', 'both_email_missing',
                  'one_email_missing']

print(f"\n========== EMAIL FEATURE SUMMARY ==========")
print(f"New features created : {new_email_cols}")
print(f"Train shape          : {train_df.shape}")
print(f"Val shape            : {val_df.shape}")
print(f"Test shape           : {test.shape}")
print(f"===========================================")


========== EMAIL FEATURE SUMMARY ==========
New features created : ['P_email_risk', 'R_email_risk', 'email_match', 'both_email_missing', 'one_email_missing']
Train shape          : (472432, 163)
Val shape            : (118108, 163)
Test shape           : (506691, 162)


STEP 6: Time Features from TransactionDT
============================================================

In [ ]:
print("TransactionDT sample values:")
print(train_df['TransactionDT'].describe())

TransactionDT sample values:
count    4.724320e+05
mean     5.715961e+06
std      3.559869e+06
min      8.640000e+04
25%      2.310158e+06
50%      5.592304e+06
75%      8.745778e+06
max      1.219284e+07
Name: TransactionDT, dtype: float64


In [ ]:
# ------------------------------------------------------------
# STEP 6a: Extract Cyclical Time Features
# Reference point is unknown but relative patterns hold
# 86400 seconds = 1 day
# 604800 seconds = 1 week
# ------------------------------------------------------------
for df in [train_df, val_df, test]:
    # Hour of day (0-23)
    df['hour_of_day'] = (df['TransactionDT'] // 3600) % 24

    # Day of week (0-6)
    df['day_of_week'] = (df['TransactionDT'] // 86400) % 7

    # Day of month approximation (1-31)
    df['day_of_month'] = (df['TransactionDT'] // 86400) % 30

    # Week number
    df['week_number'] = (df['TransactionDT'] // 604800)

print(f"\nTime features created: hour_of_day, day_of_week, day_of_month, week_number")


Time features created: hour_of_day, day_of_week, day_of_month, week_number


In [ ]:
# ------------------------------------------------------------
# STEP 6b: Fraud rate by hour of day
# Fraudsters often operate at unusual hours
# ------------------------------------------------------------
hour_fraud = train_df.groupby('hour_of_day').agg(
    count      = ('hour_of_day', 'count'),
    fraud_rate = ('isFraud', 'mean')
).reset_index()

print(f"\nFraud rate by hour of day:")
print(hour_fraud.to_string(index=False))



Fraud rate by hour of day:
 hour_of_day  count  fraud_rate
           0  30557    0.032038
           1  26772    0.032198
           2  21923    0.036993
           3  17102    0.036078
           4  12603    0.049591
           5   8392    0.066611
           6   5065    0.073840
           7   3156    0.101077
           8   2181    0.097203
           9   1844    0.105748
          10   2450    0.069388
          11   4536    0.039462
          12   8733    0.030001
          13  14977    0.021433
          14  21796    0.024500
          15  26616    0.024985
          16  30829    0.028285
          17  32774    0.032678
          18  33654    0.035033
          19  34155    0.034929
          20  33543    0.034463
          21  33583    0.033678
          22  33051    0.032162
          23  32140    0.038892


In [ ]:
# ------------------------------------------------------------
# STEP 6c: Fraud rate by day of week
# ------------------------------------------------------------
day_fraud = train_df.groupby('day_of_week').agg(
    count      = ('day_of_week', 'count'),
    fraud_rate = ('isFraud', 'mean')
).reset_index()

print(f"\nFraud rate by day of week:")
print(day_fraud.to_string(index=False))


Fraud rate by day of week:
 day_of_week  count  fraud_rate
           0  68637    0.038274
           1  80060    0.035199
           2  64926    0.036873
           3  55840    0.036909
           4  68191    0.032218
           5  66991    0.032736
           6  67787    0.034063


In [ ]:
# ------------------------------------------------------------
# STEP 6d: Is Night Transaction Flag
# Late night / early morning transactions are higher risk
# Define night as 11pm to 5am (hour 23, 0, 1, 2, 3, 4)
# ------------------------------------------------------------
for df in [train_df, val_df, test]:
    df['is_night_txn'] = df['hour_of_day'].isin(
        [23, 0, 1, 2, 3, 4]
    ).astype(np.int8)

    # Weekend flag
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(np.int8)

    # Business hours flag (9am to 5pm)
    df['is_business_hours'] = df['hour_of_day'].between(9, 17).astype(np.int8)

print(f"\nNight transaction fraud analysis:")
night_fraud = train_df.groupby('is_night_txn').agg(
    count      = ('is_night_txn', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(night_fraud)

print(f"\nWeekend fraud analysis:")
weekend_fraud = train_df.groupby('is_weekend').agg(
    count      = ('is_weekend', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(weekend_fraud)

print(f"\nBusiness hours fraud analysis:")
biz_fraud = train_df.groupby('is_business_hours').agg(
    count      = ('is_business_hours', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(biz_fraud)


Night transaction fraud analysis:
               count  fraud_rate
is_night_txn                    
0             331335    0.034572
1             141097    0.036457

Weekend fraud analysis:
             count  fraud_rate
is_weekend                    
0           337654    0.035827
1           134778    0.033403

Business hours fraud analysis:
                    count  fraud_rate
is_business_hours                    
0                  327877    0.037606
1                  144555    0.029532


In [ ]:
# ------------------------------------------------------------
# STEP 6e: Transaction Amount by Time Interaction
# Unusually large amounts at unusual times = stronger signal
# ------------------------------------------------------------
for df in [train_df, val_df, test]:
    df['night_high_amt'] = (
        (df['is_night_txn'] == 1) &
        (df['TransactionAmt'] > df['TransactionAmt'].quantile(0.75))
    ).astype(np.int8)

night_high_fraud = train_df.groupby('night_high_amt').agg(
    count      = ('night_high_amt', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(f"\nNight + High Amount interaction fraud analysis:")
print(night_high_fraud)


Night + High Amount interaction fraud analysis:
                 count  fraud_rate
night_high_amt                    
0               442263    0.034156
1                30169    0.049488


In [ ]:
print(hour_fraud.sort_values('fraud_rate', ascending=False).head(10))

In [ ]:
# ------------------------------------------------------------
# STEP 6f: Summary
# ------------------------------------------------------------
new_time_cols = ['hour_of_day', 'day_of_week', 'day_of_month',
                 'week_number', 'is_night_txn', 'is_weekend',
                 'is_business_hours', 'night_high_amt']

print(f"\n========== TIME FEATURE SUMMARY ==========")
print(f"New features created : {new_time_cols}")
print(f"Train shape          : {train_df.shape}")
print(f"Val shape            : {val_df.shape}")
print(f"Test shape           : {test.shape}")
print(f"==========================================")


========== TIME FEATURE SUMMARY ==========
New features created : ['hour_of_day', 'day_of_week', 'day_of_month', 'week_number', 'is_night_txn', 'is_weekend', 'is_business_hours', 'night_high_amt']
Train shape          : (472432, 171)
Val shape            : (118108, 171)
Test shape           : (506691, 170)


STEP 7: Transaction Velocity Features
============================================================

In [ ]:
print("Computing transaction velocity features...")
print("Sorting by TransactionDT...")

# Sort all datasets by time
train_df = train_df.sort_values('TransactionDT').reset_index(drop=True)
val_df   = val_df.sort_values('TransactionDT').reset_index(drop=True)
test     = test.sort_values('TransactionDT').reset_index(drop=True)


Computing transaction velocity features...
Sorting by TransactionDT...


In [ ]:
# ------------------------------------------------------------
# STEP 7a: Transaction count per UID in recent window
# Using expanding count grouped by UID
# This approximates "how many times has this user transacted"
# up to this point in time
# ------------------------------------------------------------

# import gc

# # Combine train and val temporarily for velocity computation
# # Rebuild only missing txn_count columns
# missing_count_cols = [uid for uid in ['UID1', 'UID2', 'UID3']
#                       if f'{uid}_txn_count' not in train_df.columns]

# print("Missing txn_count features:", missing_count_cols)

# if missing_count_cols:
#     combined = pd.concat([train_df, val_df], axis=0, ignore_index=True)
#     combined = combined.sort_values('TransactionDT').reset_index(drop=True)

#     for uid in missing_count_cols:
#         col_name = f'{uid}_txn_count'
#         print(f"Creating {col_name}...")

#         uid_temp = combined[[uid]].copy()
#         uid_temp[col_name] = uid_temp.groupby(uid, sort=False).cumcount().astype(np.int32)

#         combined[col_name] = uid_temp[col_name].to_numpy()

#         del uid_temp
#         gc.collect()

#     train_size = len(train_df)
#     train_df = combined.iloc[:train_size].copy()
#     val_df   = combined.iloc[train_size:].copy()

#     del combined
#     gc.collect()

# print("Done.")
# print([c for c in train_df.columns if 'UID1' in c])

In [ ]:
# ------------------------------------------------------------
# STEP 7b: Time since last transaction per UID
# Large gap then sudden activity = suspicious
# ------------------------------------------------------------
# for uid in ['UID1', 'UID2']:
#     print(f"\nProcessing {uid} time_since_last...")

#     uid_temp = combined[['TransactionDT', uid]].copy()

#     col_name = f'{uid}_time_since_last'
#     uid_temp[col_name] = (
#         uid_temp.groupby(uid, sort=False)['TransactionDT']
#         .diff()
#         .fillna(-1)
#         .astype(np.int32)
#     )

#     combined[col_name] = uid_temp[col_name].to_numpy()
#     print(f"  Created {col_name}")

#     del uid_temp
#     gc.collect()

In [ ]:
# ------------------------------------------------------------
# STEP 7c: Is first transaction for this UID
# First-time users have no history = higher uncertainty
# ------------------------------------------------------------
# for uid in ['UID1', 'UID2', 'UID3']:
#     print(f"\nProcessing {uid} is_first_txn...")

#     uid_temp = combined[[uid]].copy()

#     col_name = f'{uid}_is_first_txn'
#     uid_temp[col_name] = (
#         uid_temp.groupby(uid, sort=False).cumcount() == 0
#     ).astype(np.int8)

#     combined[col_name] = uid_temp[col_name].to_numpy()
#     print(f"  Created {col_name}")

#     del uid_temp
#     gc.collect()

In [ ]:
# ------------------------------------------------------------
# STEP 7d: Split combined back into train and val
# ------------------------------------------------------------
# train_size = len(train_df)

# train_df = combined.iloc[:train_size].copy()
# val_df   = combined.iloc[train_size:].copy()

# del combined
# gc.collect()

# print(f"\nSplit back:")
# print(f"  Train shape : {train_df.shape}")
# print(f"  Val shape   : {val_df.shape}")

In [ ]:
import gc
import numpy as np
import pandas as pd

print("Computing transaction velocity features...")
print("Sorting by TransactionDT...")

train_df = train_df.sort_values('TransactionDT').reset_index(drop=True)
val_df   = val_df.sort_values('TransactionDT').reset_index(drop=True)

velocity_uids = ['UID1', 'UID2', 'UID3']
time_uids = ['UID1', 'UID2']

# ------------------------------------------------------------
# Step 7a + 7b + 7c + 7d together
# ------------------------------------------------------------
print("\nComputing velocity features for train + val...")

combined = pd.concat([train_df, val_df], axis=0, ignore_index=True)
combined = combined.sort_values('TransactionDT').reset_index(drop=True)

print(f"Combined shape for velocity computation: {combined.shape}")

for uid in velocity_uids:
    print(f"\nProcessing {uid}...")

    uid_temp = combined[['TransactionDT', uid]].copy()

    # txn count
    count_col = f'{uid}_txn_count'
    uid_temp[count_col] = (
        uid_temp.groupby(uid, sort=False).cumcount().astype(np.int32)
    )

    # first transaction
    first_col = f'{uid}_is_first_txn'
    uid_temp[first_col] = (uid_temp[count_col] == 0).astype(np.int8)

    # time since last
    if uid in time_uids:
        time_col = f'{uid}_time_since_last'
        uid_temp[time_col] = (
            uid_temp.groupby(uid, sort=False)['TransactionDT']
            .diff()
            .fillna(-1)
            .astype(np.int32)
        )

    # write back
    combined[count_col] = uid_temp[count_col].to_numpy()
    combined[first_col] = uid_temp[first_col].to_numpy()

    if uid in time_uids:
        combined[time_col] = uid_temp[time_col].to_numpy()

    print(f"  Created {count_col}")
    print(f"  Created {first_col}")
    if uid in time_uids:
        print(f"  Created {time_col}")

    del uid_temp
    gc.collect()

# split back
train_size = len(train_df)
train_df = combined.iloc[:train_size].copy()
val_df   = combined.iloc[train_size:].copy()

del combined
gc.collect()

print("\nVelocity features added to train and val.")
print("Check created columns:")
for col in [
    'UID1_txn_count', 'UID2_txn_count', 'UID3_txn_count',
    'UID1_time_since_last', 'UID2_time_since_last',
    'UID1_is_first_txn', 'UID2_is_first_txn', 'UID3_is_first_txn'
]:
    print(col, col in train_df.columns)

Computing transaction velocity features...
Sorting by TransactionDT...

Computing velocity features for train + val...
Combined shape for velocity computation: (590540, 171)

Processing UID1...
  Created UID1_txn_count
  Created UID1_is_first_txn
  Created UID1_time_since_last

Processing UID2...
  Created UID2_txn_count
  Created UID2_is_first_txn
  Created UID2_time_since_last

Processing UID3...
  Created UID3_txn_count
  Created UID3_is_first_txn

Velocity features added to train and val.
Check created columns:
UID1_txn_count True
UID2_txn_count True
UID3_txn_count True
UID1_time_since_last True
UID2_time_since_last True
UID1_is_first_txn True
UID2_is_first_txn True
UID3_is_first_txn True


In [ ]:
# ------------------------------------------------------------
# STEP 7e: Apply same velocity features to test
# Test uses combined train+val history
# ------------------------------------------------------------
print("\nComputing velocity for test using train+val history...")

history_base = pd.concat(
    [
        train_df[['TransactionDT', 'UID1', 'UID2', 'UID3']],
        val_df[['TransactionDT', 'UID1', 'UID2', 'UID3']]
    ],
    axis=0,
    ignore_index=True
).sort_values('TransactionDT').reset_index(drop=True)

for uid in ['UID1', 'UID2', 'UID3']:
    print(f"\nProcessing test velocity for {uid}...")

    history_uid = history_base[['TransactionDT', uid]].copy()
    history_uid['_is_test'] = 0

    test_uid = test[['TransactionDT', uid]].copy()
    test_uid['_is_test'] = 1

    full_uid = pd.concat([history_uid, test_uid], axis=0, ignore_index=True)
    full_uid = full_uid.sort_values('TransactionDT').reset_index(drop=True)

    count_col = f'{uid}_txn_count'
    full_uid[count_col] = (
        full_uid.groupby(uid, sort=False).cumcount().astype(np.int32)
    )

    first_col = f'{uid}_is_first_txn'
    full_uid[first_col] = (
        full_uid.groupby(uid, sort=False).cumcount() == 0
    ).astype(np.int8)

    test_mask = full_uid['_is_test'] == 1
    test[count_col] = full_uid.loc[test_mask, count_col].to_numpy()
    test[first_col] = full_uid.loc[test_mask, first_col].to_numpy()

    if uid in ['UID1', 'UID2']:
        time_col = f'{uid}_time_since_last'
        full_uid[time_col] = (
            full_uid.groupby(uid, sort=False)['TransactionDT']
            .diff()
            .fillna(-1)
            .astype(np.int32)
        )
        test[time_col] = full_uid.loc[test_mask, time_col].to_numpy()

    del history_uid, test_uid, full_uid
    gc.collect()

del history_base
gc.collect()

print(f"  Test shape  : {test.shape}")


Computing velocity for test using train+val history...

Processing test velocity for UID1...

Processing test velocity for UID2...

Processing test velocity for UID3...
  Test shape  : (506691, 178)


In [ ]:
# ------------------------------------------------------------
# STEP 7f: Fraud rate analysis on velocity features
# ------------------------------------------------------------
print(f"\nVelocity fraud analysis:")

# First transaction fraud rate
print(f"\nUID1 first transaction:")
uid1_first = train_df.groupby('UID1_is_first_txn').agg(
    count      = ('UID1_is_first_txn', 'count'),
    fraud_rate = ('isFraud', 'mean')
)
print(uid1_first)

# Transaction count vs fraud
print(f"\nUID1 transaction count buckets vs fraud rate:")
required_cols = ['UID1_txn_count', 'UID1_time_since_last', 'UID1_is_first_txn']
missing = [c for c in required_cols if c not in train_df.columns]

if missing:
    print("Step 7f skipped. Missing columns:", missing)
else:
    print(f"\nVelocity fraud analysis:")

    print(f"\nUID1 first transaction:")
    uid1_first = train_df.groupby('UID1_is_first_txn').agg(
        count=('UID1_is_first_txn', 'count'),
        fraud_rate=('isFraud', 'mean')
    )
    print(uid1_first)

    print(f"\nUID1 transaction count buckets vs fraud rate:")
    train_df['UID1_count_bucket'] = pd.cut(
        train_df['UID1_txn_count'],
        bins=[-1, 0, 1, 5, 10, 50, float('inf')],
        labels=['0', '1', '2-5', '6-10', '11-50', '50+']
    )
    count_fraud = train_df.groupby('UID1_count_bucket', observed=True).agg(
        count=('UID1_count_bucket', 'count'),
        fraud_rate=('isFraud', 'mean')
    )
    print(count_fraud)

    print(f"\nUID1 time since last transaction:")
    train_df['UID1_time_bucket'] = pd.cut(
        train_df['UID1_time_since_last'],
        bins=[-2, 0, 3600, 86400, 604800, float('inf')],
        labels=['first_txn', 'within_1h', 'within_1d', 'within_1w', 'over_1w']
    )
    time_fraud = train_df.groupby('UID1_time_bucket', observed=True).agg(
        count=('UID1_time_bucket', 'count'),
        fraud_rate=('isFraud', 'mean')
    )
    print(time_fraud)

    train_df.drop(columns=['UID1_count_bucket', 'UID1_time_bucket'], inplace=True)


Velocity fraud analysis:

UID1 first transaction:
                    count  fraud_rate
UID1_is_first_txn                    
0                  458974    0.035455
1                   13458    0.024224

UID1 transaction count buckets vs fraud rate:

Velocity fraud analysis:

UID1 first transaction:
                    count  fraud_rate
UID1_is_first_txn                    
0                  458974    0.035455
1                   13458    0.024224

UID1 transaction count buckets vs fraud rate:
                    count  fraud_rate
UID1_count_bucket                    
0                   13458    0.024224
1                    9665    0.025142
2-5                 25973    0.025334
6-10                20797    0.026350
11-50               69785    0.030995
50+                332754    0.038049

UID1 time since last transaction:
                   count  fraud_rate
UID1_time_bucket                    
first_txn          13609    0.024837
within_1h         199278    0.041891
within_1d    

In [ ]:
# ------------------------------------------------------------
# STEP 7g: Summary
# ------------------------------------------------------------
velocity_cols = [f'{uid}_txn_count'      for uid in ['UID1','UID2','UID3']] + \
                [f'{uid}_time_since_last' for uid in ['UID1','UID2']]       + \
                [f'{uid}_is_first_txn'    for uid in ['UID1','UID2','UID3']]

print(f"\n========== VELOCITY FEATURE SUMMARY ==========")
print(f"New features created : {len(velocity_cols)}")
print(f"{velocity_cols}")
print(f"Train shape          : {train_df.shape}")
print(f"Val shape            : {val_df.shape}")
print(f"Test shape           : {test.shape}")
print(f"===============================================")


========== VELOCITY FEATURE SUMMARY ==========
New features created : 8
['UID1_txn_count', 'UID2_txn_count', 'UID3_txn_count', 'UID1_time_since_last', 'UID2_time_since_last', 'UID1_is_first_txn', 'UID2_is_first_txn', 'UID3_is_first_txn']
Train shape          : (472432, 179)
Val shape            : (118108, 179)
Test shape           : (506691, 178)


Freezing Until Step 7
============================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

In [ ]:
os.makedirs(SAVE_PATH, exist_ok=True)

print("Freezing pipeline after Step 7...")

# Check Step 7 columns
step7_required = [
    'UID1_txn_count', 'UID2_txn_count', 'UID3_txn_count',
    'UID1_is_first_txn', 'UID2_is_first_txn', 'UID3_is_first_txn',
    'UID1_time_since_last', 'UID2_time_since_last'
]

missing_step7 = [c for c in step7_required if c not in train_df.columns]
if missing_step7:
    print("WARNING: Missing Step 7 columns:", missing_step7)
else:
    print("All key Step 7 columns found.")

# Memory check
print("\nMemory usage before save:")
print(f"Train: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Val  : {val_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Test : {test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# SAVE TO GOOGLE DRIVE
train_df.to_csv(SAVE_PATH + 'train_step7_frozen.csv', index=False)
val_df.to_csv(SAVE_PATH + 'val_step7_frozen.csv', index=False)
test.to_csv(SAVE_PATH + 'test_step7_frozen.csv', index=False)

print("\nSaved to Google Drive:")
print(SAVE_PATH)

# Metadata file
with open(SAVE_PATH + "step7_freeze_note.txt", "w") as f:
    f.write("Checkpoint saved after Step 7 velocity features.\n")
    f.write(f"Train shape: {train_df.shape}\n")
    f.write(f"Val shape: {val_df.shape}\n")
    f.write(f"Test shape: {test.shape}\n")

print("\nFreeze complete.")

RELOAD FROM STEP 7 FROZEN CHECKPOINT
============================================================

In [1]:
import gc
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

print("Reloading frozen Step 7 files...")

train_df = pd.read_csv(SAVE_PATH + 'train_step7_frozen.csv')
val_df   = pd.read_csv(SAVE_PATH + 'val_step7_frozen.csv')
test     = pd.read_csv(SAVE_PATH +'test_step7_frozen.csv')

print("Reload complete.")
print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test.shape)

gc.collect()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reloading frozen Step 7 files...
Reload complete.
Train shape: (472432, 179)
Val shape  : (118108, 179)
Test shape : (506691, 178)


0

STEP 8: Aggregation Features
============================================================

In [2]:
uid_cols = ['UID1', 'UID2', 'UID3', 'UID4']
print(f"\nUID cols confirmed: {[c for c in uid_cols if c in train_df.columns]}")
print(f"card1 exists: {'card1' in train_df.columns}")
print(f"TransactionAmt exists: {'TransactionAmt' in train_df.columns}")


UID cols confirmed: ['UID1', 'UID2', 'UID3', 'UID4']
card1 exists: True
TransactionAmt exists: True


In [3]:
# ============================================================
# STEP 8a: Aggregation Features Per UID
# Train-only statistics, map outward
# ============================================================
print("\nComputing aggregation features per UID...")

agg_cols_created = []

# Global fallback values from TRAIN only
global_mean_amt = np.float32(train_df['TransactionAmt'].mean())
global_std_amt  = np.float32(train_df['TransactionAmt'].std())
global_max_amt  = np.float32(train_df['TransactionAmt'].max())
global_count_amt = np.int32(1)

for uid in ['UID1', 'UID2', 'UID3']:
    print(f"\n  Processing {uid}...")

    # Narrow train-only source
    uid_stats = (
        train_df[[uid, 'TransactionAmt']]
        .groupby(uid, dropna=False)['TransactionAmt']
        .agg(['mean', 'std', 'count', 'max'])
        .reset_index()
        .rename(columns={
            'mean':  f'{uid}_amt_mean',
            'std':   f'{uid}_amt_std',
            'count': f'{uid}_amt_count',
            'max':   f'{uid}_amt_max'
        })
    )

    # Fill one-transaction std = 0
    uid_stats[f'{uid}_amt_std'] = uid_stats[f'{uid}_amt_std'].fillna(0)

    # Build maps
    mean_map  = uid_stats.set_index(uid)[f'{uid}_amt_mean']
    std_map   = uid_stats.set_index(uid)[f'{uid}_amt_std']
    count_map = uid_stats.set_index(uid)[f'{uid}_amt_count']
    max_map   = uid_stats.set_index(uid)[f'{uid}_amt_max']

    # Apply with map
    for df in [train_df, val_df, test]:
        df[f'{uid}_amt_mean']  = df[uid].map(mean_map)
        df[f'{uid}_amt_std']   = df[uid].map(std_map)
        df[f'{uid}_amt_count'] = df[uid].map(count_map)
        df[f'{uid}_amt_max']   = df[uid].map(max_map)

        # Fill unseen UIDs with train-global fallback
        df[f'{uid}_amt_mean']  = df[f'{uid}_amt_mean'].fillna(global_mean_amt)
        df[f'{uid}_amt_std']   = df[f'{uid}_amt_std'].fillna(global_std_amt)
        df[f'{uid}_amt_count'] = df[f'{uid}_amt_count'].fillna(global_count_amt)
        df[f'{uid}_amt_max']   = df[f'{uid}_amt_max'].fillna(global_max_amt)

        # Raw deviation from mean
        df[f'{uid}_amt_dev'] = (
            df['TransactionAmt'] - df[f'{uid}_amt_mean']
        )

        # Ratio to mean
        df[f'{uid}_amt_ratio'] = (
            df['TransactionAmt'] / (df[f'{uid}_amt_mean'] + 1)
        )

        # Z-score style deviation
        df[f'{uid}_amt_zscore'] = (
            (df['TransactionAmt'] - df[f'{uid}_amt_mean']) /
            (df[f'{uid}_amt_std'] + 1)
        )

        # Safe downcast
        float_cols = [
            f'{uid}_amt_mean',
            f'{uid}_amt_std',
            f'{uid}_amt_max',
            f'{uid}_amt_dev',
            f'{uid}_amt_ratio',
            f'{uid}_amt_zscore'
        ]
        for col in float_cols:
            df[col] = df[col].astype(np.float32)

        df[f'{uid}_amt_count'] = df[f'{uid}_amt_count'].astype(np.int32)

    new_cols = [
        f'{uid}_amt_mean',
        f'{uid}_amt_std',
        f'{uid}_amt_count',
        f'{uid}_amt_max',
        f'{uid}_amt_dev',
        f'{uid}_amt_ratio',
        f'{uid}_amt_zscore'
    ]
    agg_cols_created.extend(new_cols)
    print(f"    Created: {new_cols}")

    del uid_stats, mean_map, std_map, count_map, max_map
    gc.collect()


Computing aggregation features per UID...

  Processing UID1...
    Created: ['UID1_amt_mean', 'UID1_amt_std', 'UID1_amt_count', 'UID1_amt_max', 'UID1_amt_dev', 'UID1_amt_ratio', 'UID1_amt_zscore']

  Processing UID2...
    Created: ['UID2_amt_mean', 'UID2_amt_std', 'UID2_amt_count', 'UID2_amt_max', 'UID2_amt_dev', 'UID2_amt_ratio', 'UID2_amt_zscore']

  Processing UID3...
    Created: ['UID3_amt_mean', 'UID3_amt_std', 'UID3_amt_count', 'UID3_amt_max', 'UID3_amt_dev', 'UID3_amt_ratio', 'UID3_amt_zscore']


In [4]:
# ============================================================
# STEP 8b: Aggregation Features Per card1
# ============================================================
print("\nComputing aggregation features per card1...")

card1_stats = (
    train_df[['card1', 'TransactionAmt']]
    .groupby('card1', dropna=False)['TransactionAmt']
    .agg(['mean', 'std', 'count', 'max'])
    .reset_index()
    .rename(columns={
        'mean':  'card1_amt_mean',
        'std':   'card1_amt_std',
        'count': 'card1_txn_count',
        'max':   'card1_amt_max'
    })
)

card1_stats['card1_amt_std'] = card1_stats['card1_amt_std'].fillna(0)

card1_mean_map  = card1_stats.set_index('card1')['card1_amt_mean']
card1_std_map   = card1_stats.set_index('card1')['card1_amt_std']
card1_count_map = card1_stats.set_index('card1')['card1_txn_count']
card1_max_map   = card1_stats.set_index('card1')['card1_amt_max']

for df in [train_df, val_df, test]:
    df['card1_amt_mean']  = df['card1'].map(card1_mean_map)
    df['card1_amt_std']   = df['card1'].map(card1_std_map)
    df['card1_txn_count'] = df['card1'].map(card1_count_map)
    df['card1_amt_max']   = df['card1'].map(card1_max_map)

    df['card1_amt_mean']  = df['card1_amt_mean'].fillna(global_mean_amt)
    df['card1_amt_std']   = df['card1_amt_std'].fillna(global_std_amt)
    df['card1_txn_count'] = df['card1_txn_count'].fillna(global_count_amt)
    df['card1_amt_max']   = df['card1_amt_max'].fillna(global_max_amt)

    df['card1_amt_dev'] = df['TransactionAmt'] - df['card1_amt_mean']
    df['card1_amt_ratio'] = df['TransactionAmt'] / (df['card1_amt_mean'] + 1)

    for col in ['card1_amt_mean', 'card1_amt_std', 'card1_amt_max',
                'card1_amt_dev', 'card1_amt_ratio']:
        df[col] = df[col].astype(np.float32)

    df['card1_txn_count'] = df['card1_txn_count'].astype(np.int32)

agg_cols_created.extend([
    'card1_amt_mean', 'card1_amt_std', 'card1_txn_count',
    'card1_amt_max', 'card1_amt_dev', 'card1_amt_ratio'
])

print("  Created card1_amt_mean, card1_amt_std, card1_txn_count, card1_amt_max, card1_amt_dev, card1_amt_ratio")

del card1_stats, card1_mean_map, card1_std_map, card1_count_map, card1_max_map
gc.collect()


Computing aggregation features per card1...
  Created card1_amt_mean, card1_amt_std, card1_txn_count, card1_amt_max, card1_amt_dev, card1_amt_ratio


0

In [5]:
# ============================================================
# STEP 8c: Optional verification block
# Keep this because RAM is still acceptable unless it spikes
# ============================================================
print("\nVerifying aggregation fraud signals...")

train_df['amt_ratio_bucket'] = pd.cut(
    train_df['UID1_amt_ratio'],
    bins=[0, 0.5, 1.0, 2.0, 5.0, float('inf')],
    labels=['much_less', 'less', 'normal', 'more', 'much_more']
)

ratio_fraud = train_df.groupby('amt_ratio_bucket', observed=True).agg(
    count=('amt_ratio_bucket', 'count'),
    fraud_rate=('isFraud', 'mean')
).sort_values('fraud_rate', ascending=False)

print("\nUID1 amount ratio vs fraud rate:")
print(ratio_fraud)

train_df['card1_count_bucket'] = pd.cut(
    train_df['card1_txn_count'],
    bins=[0, 1, 5, 20, 100, float('inf')],
    labels=['1', '2-5', '6-20', '21-100', '100+']
)

count_fraud = train_df.groupby('card1_count_bucket', observed=True).agg(
    count=('card1_count_bucket', 'count'),
    fraud_rate=('isFraud', 'mean')
).sort_values('fraud_rate', ascending=False)

print("\ncard1 transaction count vs fraud rate:")
print(count_fraud)

train_df.drop(columns=['amt_ratio_bucket', 'card1_count_bucket'], inplace=True)



Verifying aggregation fraud signals...

UID1 amount ratio vs fraud rate:
                   count  fraud_rate
amt_ratio_bucket                    
more               37265    0.053884
normal             92238    0.045589
much_more           7359    0.034516
less              154110    0.033768
much_less         181460    0.027158

card1 transaction count vs fraud rate:
                     count  fraud_rate
card1_count_bucket                    
1                     3380    0.038462
100+                356844    0.037537
21-100               66361    0.029159
2-5                  12695    0.027727
6-20                 33152    0.023739


In [6]:
# ============================================================
# STEP 8d: Summary
# ============================================================
gc.collect()

print("\n========== AGGREGATION FEATURE SUMMARY ==========")
print(f"Total new features created : {len(agg_cols_created)}")
print(f"Features: {agg_cols_created}")
print("\nNull check after aggregation:")
print(f"  Train nulls : {train_df.isnull().sum().sum()}")
print(f"  Val nulls   : {val_df.isnull().sum().sum()}")
print(f"  Test nulls  : {test.isnull().sum().sum()}")
print("\nShapes:")
print(f"  Train : {train_df.shape}")
print(f"  Val   : {val_df.shape}")
print(f"  Test  : {test.shape}")
print("\nMemory usage:")
print(f"  Train : {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Val   : {val_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Test  : {test.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("==================================================")


========== AGGREGATION FEATURE SUMMARY ==========
Total new features created : 27
Features: ['UID1_amt_mean', 'UID1_amt_std', 'UID1_amt_count', 'UID1_amt_max', 'UID1_amt_dev', 'UID1_amt_ratio', 'UID1_amt_zscore', 'UID2_amt_mean', 'UID2_amt_std', 'UID2_amt_count', 'UID2_amt_max', 'UID2_amt_dev', 'UID2_amt_ratio', 'UID2_amt_zscore', 'UID3_amt_mean', 'UID3_amt_std', 'UID3_amt_count', 'UID3_amt_max', 'UID3_amt_dev', 'UID3_amt_ratio', 'UID3_amt_zscore', 'card1_amt_mean', 'card1_amt_std', 'card1_txn_count', 'card1_amt_max', 'card1_amt_dev', 'card1_amt_ratio']

Null check after aggregation:
  Train nulls : 0
  Val nulls   : 0
  Test nulls  : 0

Shapes:
  Train : (472432, 206)
  Val   : (118108, 206)
  Test  : (506691, 205)

Memory usage:
  Train : 1477.9 MB
  Val   : 368.5 MB
  Test  : 1578.3 MB


STEP 9: Interaction + Simple Features
============================================================

In [7]:
print("Starting Step 9: Interaction + Simple Features...")

step9_cols_created = []

Starting Step 9: Interaction + Simple Features...


In [8]:
# ------------------------------------------------------------
# STEP 9a: Amount roundness features
# ------------------------------------------------------------
print("\nCreating amount roundness features...")

for df in [train_df, val_df, test]:
    # Exact integer amount
    df['amt_is_integer'] = (np.floor(df['TransactionAmt']) == df['TransactionAmt']).astype(np.int8)

    # Rounded to 1 decimal place
    df['amt_is_1dp_round'] = (
        np.round(df['TransactionAmt'], 1) == df['TransactionAmt']
    ).astype(np.int8)

    # Large round amount (more suspicious than tiny round values)
    df['amt_is_large_round'] = (
        (df['TransactionAmt'] >= 100) &
        (np.floor(df['TransactionAmt']) == df['TransactionAmt'])
    ).astype(np.int8)

step9_cols_created.extend([
    'amt_is_integer',
    'amt_is_1dp_round',
    'amt_is_large_round'
])

print("Created: amt_is_integer, amt_is_1dp_round, amt_is_large_round")


Creating amount roundness features...
Created: amt_is_integer, amt_is_1dp_round, amt_is_large_round


In [9]:
# ------------------------------------------------------------
# STEP 9b: card1 + addr1 interaction
# train-only frequency + first-seen flag
# ------------------------------------------------------------
print("\nCreating card1 + addr1 interaction features...")

for df in [train_df, val_df, test]:
    df['card1_addr1_combo'] = (
        df['card1'].astype(str) + '_' + df['addr1'].astype(str)
    )

combo_counts = train_df['card1_addr1_combo'].value_counts(dropna=False)

for df in [train_df, val_df, test]:
    df['card1_addr1_count'] = df['card1_addr1_combo'].map(combo_counts).fillna(0).astype(np.int32)
    df['is_new_card1_addr1'] = (df['card1_addr1_count'] == 0).astype(np.int8)

step9_cols_created.extend([
    'card1_addr1_combo',
    'card1_addr1_count',
    'is_new_card1_addr1'
])

print("Created: card1_addr1_combo, card1_addr1_count, is_new_card1_addr1")

del combo_counts
gc.collect()


Creating card1 + addr1 interaction features...
Created: card1_addr1_combo, card1_addr1_count, is_new_card1_addr1


0

In [10]:
# ------------------------------------------------------------
# STEP 9c: email + device interaction
# use P_emaildomain + DeviceType
# train-only frequency + first-seen flag
# ------------------------------------------------------------
print("\nCreating email + device interaction features...")

for df in [train_df, val_df, test]:
    df['email_device_combo'] = (
        df['P_emaildomain'].astype(str) + '_' + df['DeviceType'].astype(str)
    )

email_device_counts = train_df['email_device_combo'].value_counts(dropna=False)

for df in [train_df, val_df, test]:
    df['email_device_count'] = df['email_device_combo'].map(email_device_counts).fillna(0).astype(np.int32)
    df['is_new_email_device'] = (df['email_device_count'] == 0).astype(np.int8)

step9_cols_created.extend([
    'email_device_combo',
    'email_device_count',
    'is_new_email_device'
])

print("Created: email_device_combo, email_device_count, is_new_email_device")

del email_device_counts
gc.collect()


Creating email + device interaction features...
Created: email_device_combo, email_device_count, is_new_email_device


0

In [11]:
# ------------------------------------------------------------
# STEP 9d: Optional quick verification
# ------------------------------------------------------------
print("\nQuick verification of Step 9 signals...")

# Large round amount vs fraud
round_fraud = train_df.groupby('amt_is_large_round').agg(
    count=('amt_is_large_round', 'count'),
    fraud_rate=('isFraud', 'mean')
)
print("\nLarge round amount vs fraud:")
print(round_fraud)

# New card1+addr1 combo vs fraud
card_addr_fraud = train_df.groupby('is_new_card1_addr1').agg(
    count=('is_new_card1_addr1', 'count'),
    fraud_rate=('isFraud', 'mean')
)
print("\nNew card1+addr1 combo vs fraud:")
print(card_addr_fraud)

# New email+device combo vs fraud
email_device_fraud = train_df.groupby('is_new_email_device').agg(
    count=('is_new_email_device', 'count'),
    fraud_rate=('isFraud', 'mean')
)
print("\nNew email+device combo vs fraud:")
print(email_device_fraud)


Quick verification of Step 9 signals...

Large round amount vs fraud:
                     count  fraud_rate
amt_is_large_round                    
0                   355560    0.030993
1                   116872    0.047736

New card1+addr1 combo vs fraud:
                     count  fraud_rate
is_new_card1_addr1                    
0                   472432    0.035135

New email+device combo vs fraud:
                      count  fraud_rate
is_new_email_device                    
0                    472432    0.035135


In [12]:
# ------------------------------------------------------------
# STEP 9e: Summary
# ------------------------------------------------------------
gc.collect()

print("\n========== STEP 9 FEATURE SUMMARY ==========")
print(f"Total new features created : {len(step9_cols_created)}")
print(f"Features: {step9_cols_created}")
print("\nNull check after Step 9:")
print(f"  Train nulls : {train_df.isnull().sum().sum()}")
print(f"  Val nulls   : {val_df.isnull().sum().sum()}")
print(f"  Test nulls  : {test.isnull().sum().sum()}")
print("\nShapes:")
print(f"  Train : {train_df.shape}")
print(f"  Val   : {val_df.shape}")
print(f"  Test  : {test.shape}")
print("\nMemory usage:")
print(f"  Train : {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Val   : {val_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Test  : {test.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("============================================")


========== STEP 9 FEATURE SUMMARY ==========
Total new features created : 9
Features: ['amt_is_integer', 'amt_is_1dp_round', 'amt_is_large_round', 'card1_addr1_combo', 'card1_addr1_count', 'is_new_card1_addr1', 'email_device_combo', 'email_device_count', 'is_new_email_device']

Null check after Step 9:
  Train nulls : 0
  Val nulls   : 0
  Test nulls  : 0

Shapes:
  Train : (472432, 215)
  Val   : (118108, 215)
  Test  : (506691, 214)

Memory usage:
  Train : 1540.3 MB
  Val   : 384.1 MB
  Test  : 1645.3 MB


STEP 10: Quick CatBoost Importance Check
============================================================

In [13]:
import gc
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 10: Quick CatBoost Importance Check
# Conservative feature screening only
# ============================================================
# Install CatBoost if not already installed
!pip install catboost

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
print("Starting Step 10: Quick CatBoost Importance Check...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00
Starting Step 10: Quick CatBoost Importance Check...


In [14]:
# ------------------------------------------------------------
# STEP 10a: Remove obvious non-feature columns
# ------------------------------------------------------------
exclude_cols = ['isFraud']

if 'TransactionID' in train_df.columns:
    exclude_cols.append('TransactionID')

# Remove any leftover temporary bucket columns if they somehow exist
temp_patterns = ['bucket']
for col in train_df.columns:
    if any(pat in col.lower() for pat in temp_patterns):
        exclude_cols.append(col)

exclude_cols = sorted(list(set(exclude_cols)))
print("\nExcluded columns:")
print(exclude_cols)

# Candidate feature list
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Remove constant columns
constant_cols = [c for c in feature_cols if train_df[c].nunique(dropna=False) <= 1]
if constant_cols:
    print(f"\nDropping constant columns: {len(constant_cols)}")
    feature_cols = [c for c in feature_cols if c not in constant_cols]

print(f"\nTotal candidate features: {len(feature_cols)}")


Excluded columns:
['TransactionID', 'isFraud']

Dropping constant columns: 2

Total candidate features: 211


In [15]:
# ------------------------------------------------------------
# STEP 10b: Identify categorical columns for CatBoost
# ------------------------------------------------------------
cat_features = [c for c in feature_cols if train_df[c].dtype == 'object']

print(f"Categorical feature count: {len(cat_features)}")
print("Sample categorical columns:")
print(cat_features[:20])

Categorical feature count: 38
Sample categorical columns:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30']


In [16]:
# ------------------------------------------------------------
# STEP 10c: Build train / val matrices
# Keep time split intact
# ------------------------------------------------------------
X_train = train_df[feature_cols].copy()
y_train = train_df['isFraud'].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df['isFraud'].copy()

print("\nTrain matrix:", X_train.shape)
print("Val matrix  :", X_val.shape)


Train matrix: (472432, 211)
Val matrix  : (118108, 211)


In [17]:
# ------------------------------------------------------------
# STEP 10d: Quick CatBoost model
# Light model for feature screening, not final tuning
# ------------------------------------------------------------
model = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=20
)

print("\nTraining quick CatBoost model...")
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)


Training quick CatBoost model...
0:	test: 0.7004119	best: 0.7004119 (0)	total: 2.59s	remaining: 4m 16s
20:	test: 0.8793037	best: 0.8793037 (20)	total: 51.8s	remaining: 3m 15s
40:	test: 0.8889506	best: 0.8890092 (39)	total: 1m 17s	remaining: 1m 52s
60:	test: 0.8932308	best: 0.8932308 (60)	total: 1m 46s	remaining: 1m 8s
80:	test: 0.8966229	best: 0.8967210 (79)	total: 2m 15s	remaining: 31.7s
99:	test: 0.8986956	best: 0.8988027 (98)	total: 2m 42s	remaining: 0us

bestTest = 0.8988026838
bestIteration = 98

Shrink model to first 99 iterations.


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=100, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=20)

In [18]:
# ------------------------------------------------------------
# STEP 10e: Validation check
# ------------------------------------------------------------
val_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)

print(f"\nQuick CatBoost validation AUC: {val_auc:.5f}")


Quick CatBoost validation AUC: 0.89880


In [19]:
# ------------------------------------------------------------
# STEP 10f: Feature importance extraction
# ------------------------------------------------------------
importances = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("\nTop 30 features:")
print(importance_df.head(30))

zero_importance_features = importance_df.loc[
    importance_df['importance'] == 0, 'feature'
].tolist()

print(f"\nZero-importance feature count: {len(zero_importance_features)}")
print("Zero-importance features:")
print(zero_importance_features[:50])


Top 30 features:
           feature  importance
0             UID3   16.225229
1              C13    6.612553
2              C14    6.599393
3               C1    6.541378
4             V308    4.430212
5               M5    2.774718
6             V307    2.661128
7     UID3_amt_std    2.370806
8               C2    2.212356
9             UID2    2.080975
10  UID3_amt_count    1.700500
11   TransactionDT    1.598636
12              C7    1.546543
13      DeviceInfo    1.489199
14  amt_is_integer    1.410438
15              D2    1.394166
16             V53    1.369388
17            V133    1.297207
18           id_29    1.114482
19              D8    1.103580
20            V263    1.078116
21     week_number    1.073323
22             C11    1.034735
23            V134    1.004231
24              D1    0.997706
25   R_emaildomain    0.956018
26            V281    0.923010
27  UID2_txn_count    0.919056
28              C4    0.916876
29            V306    0.884262

Zero-importance feat

In [20]:
# ------------------------------------------------------------
# STEP 10g: Conservative pruning
# Controlled Feature Pruning
# ------------------------------------------------------------
# ================================
# 1. DEFINE DROP CANDIDATES
# Based on latest zero-importance output
# ================================
zero_importance_candidates = [
    'card4', 'ProductCD', 'card3',
    'V164', 'V166', 'V202', 'V136', 'V95', 'V1',
    'V129', 'V131', 'V130', 'V132', 'V135', 'V35', 'V126',
    'D9', 'M3', 'M2', 'M7', 'C9', 'D11', 'M1', 'D14', 'M8', 'M9',
    'card1', 'card2', 'dist1', 'C3',
    'V214', 'V203', 'V204', 'V207', 'V159',
    'id_33_was_missing', 'id_36', 'id_37', 'id_32',
    'id_31_was_missing', 'id_30_was_missing',
    'DeviceInfo_was_missing', 'DeviceType_was_missing',
    'id_35', 'id_19', 'id_28',
    'V309', 'V316', 'V312', 'V313'
]


In [21]:
# ================================
# 1. DEFINE DROP CANDIDATES
# Based on latest zero-importance output
# ================================
zero_importance_candidates = [
    'card4', 'ProductCD', 'card3',
    'V164', 'V166', 'V202', 'V136', 'V95', 'V1',
    'V129', 'V131', 'V130', 'V132', 'V135', 'V35', 'V126',
    'D9', 'M3', 'M2', 'M7', 'C9', 'D11', 'M1', 'D14', 'M8', 'M9',
    'card1', 'card2', 'dist1', 'C3',
    'V214', 'V203', 'V204', 'V207', 'V159',
    'id_33_was_missing', 'id_36', 'id_37', 'id_32',
    'id_31_was_missing', 'id_30_was_missing',
    'DeviceInfo_was_missing', 'DeviceType_was_missing',
    'id_35', 'id_19', 'id_28',
    'V309', 'V316', 'V312', 'V313'
]


In [22]:
# ================================
# 2. PROTECT IMPORTANT FEATURES
# Keep these even if zero in quick CatBoost
# ================================
protected_cols = [
    'isFraud',
    'TransactionID',
    'TransactionDT',
    'TransactionAmt',
    'UID1', 'UID2', 'UID3', 'UID4',
    'card1', 'card2', 'card3', 'card4',
    'ProductCD',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'DeviceType', 'DeviceInfo'
]

drop_cols = [c for c in zero_importance_candidates if c not in protected_cols]

print(f"Total zero-importance candidates: {len(zero_importance_candidates)}")
print(f"Protected columns kept: {[c for c in zero_importance_candidates if c in protected_cols]}")
print(f"Final columns to drop: {len(drop_cols)}")
print(drop_cols)

Total zero-importance candidates: 50
Protected columns kept: ['card4', 'ProductCD', 'card3', 'card1', 'card2']
Final columns to drop: 45
['V164', 'V166', 'V202', 'V136', 'V95', 'V1', 'V129', 'V131', 'V130', 'V132', 'V135', 'V35', 'V126', 'D9', 'M3', 'M2', 'M7', 'C9', 'D11', 'M1', 'D14', 'M8', 'M9', 'dist1', 'C3', 'V214', 'V203', 'V204', 'V207', 'V159', 'id_33_was_missing', 'id_36', 'id_37', 'id_32', 'id_31_was_missing', 'id_30_was_missing', 'DeviceInfo_was_missing', 'DeviceType_was_missing', 'id_35', 'id_19', 'id_28', 'V309', 'V316', 'V312', 'V313']


In [23]:
# ================================
# 3. APPLY DROP
# ================================
train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
val_df   = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
test     = test.drop(columns=[c for c in drop_cols if c in test.columns])

In [24]:
# ================================
# 4. SUMMARY
# ================================
print("\nAfter Step 10G pruning:")
print(f"Train shape: {train_df.shape}")
print(f"Val shape  : {val_df.shape}")
print(f"Test shape : {test.shape}")

print("\nMemory usage after pruning:")
print(f"Train: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Val  : {val_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Test : {test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nStep 10G complete.")


After Step 10G pruning:
Train shape: (472432, 170)
Val shape  : (118108, 170)
Test shape : (506691, 169)

Memory usage after pruning:
Train: 1171.21 MB
Val  : 292.61 MB
Test : 1252.44 MB

Step 10G complete.


In [ ]:
# ============================================================
# FREEZE AFTER STEP 10G
# ============================================================

print("Freezing dataset after Step 10G...")

SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

# Save datasets
train_df.to_csv(SAVE_PATH + 'train_step10_pruned.csv', index=False)
val_df.to_csv(SAVE_PATH + 'val_step10_pruned.csv', index=False)
test.to_csv(SAVE_PATH + 'test_step10_pruned.csv', index=False)

# Save feature list
feature_cols = [c for c in train_df.columns if c != 'isFraud']

pd.DataFrame({'feature': feature_cols}).to_csv(
    SAVE_PATH + 'step10_final_feature_list.csv',
    index=False
)

# Save dropped features
pd.DataFrame({'dropped_feature': drop_cols}).to_csv(
    SAVE_PATH + 'step10_dropped_features.csv',
    index=False
)

print("\nSaved:")
print(" - train_step10_pruned.csv")
print(" - val_step10_pruned.csv")
print(" - test_step10_pruned.csv")
print(" - step10_final_feature_list.csv")
print(" - step10_dropped_features.csv")

print("\nFreeze complete.")

In [26]:
# ------------------------------------------------------------
# STEP 10h: Save importance results and reduced feature list
# ------------------------------------------------------------

# Define reduced_feature_cols from the current state of train_df
importance_df.to_csv(SAVE_PATH + 'step10_feature_importance.csv', index=False)

pd.DataFrame({'feature': reduced_feature_cols}).to_csv(
    SAVE_PATH + 'step10_reduced_feature_list.csv', index=False
)

pd.DataFrame({'feature': zero_importance_features}).to_csv(
    SAVE_PATH + 'step10_zero_importance_features.csv', index=False
)

print("\nSaved:")
print(" - step10_feature_importance.csv")
print(" - step10_reduced_feature_list.csv")
print(" - step10_zero_importance_features.csv")


Saved:
 - step10_feature_importance.csv
 - step10_reduced_feature_list.csv
 - step10_zero_importance_features.csv


STEP 11: Hybrid Prep Pipeline
============================================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# STEP 11A: CatBoost Prep Pipeline
# Load from Step 10 pruned files
# Keep categoricals as strings
# Use class weights instead of SMOTE
# Save CatBoost-ready files + metadata
# ============================================================

import gc
import os
import pickle
import pandas as pd
import numpy as np
from collections import Counter

SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

print("Loading Step 10 pruned files for CatBoost prep...")

train_df = pd.read_csv(SAVE_PATH + 'train_step10_pruned.csv')
val_df   = pd.read_csv(SAVE_PATH + 'val_step10_pruned.csv')
test     = pd.read_csv(SAVE_PATH + 'test_step10_pruned.csv')

exclude_cols = ['isFraud']
if 'TransactionID' in train_df.columns:
    exclude_cols.append('TransactionID')

feature_cols = [c for c in train_df.columns if c not in exclude_cols]
cat_features = [c for c in feature_cols if train_df[c].dtype == 'object']
num_features = [c for c in feature_cols if c not in cat_features]

print(f"Feature cols     : {len(feature_cols)}")
print(f"Categorical cols : {len(cat_features)}")
print(f"Numeric cols     : {len(num_features)}")

# Keep categoricals as strings for CatBoost consistency
for col in cat_features:
    train_df[col] = train_df[col].astype(str)
    val_df[col]   = val_df[col].astype(str)
    test[col]     = test[col].astype(str)

# Compute class weights from training distribution
y_train = train_df['isFraud'].values
class_counts = Counter(y_train)

neg_count = class_counts[0]
pos_count = class_counts[1]

# Simple inverse-frequency style weights
class_weights = {
    0: 1.0,
    1: float(neg_count / pos_count)
}

print("\nTraining class distribution:")
print(class_counts)
print("Class weights for CatBoost:")
print(class_weights)

# Save CatBoost-ready datasets
train_df.to_csv(SAVE_PATH + 'train_catboost_ready.csv', index=False)
val_df.to_csv(SAVE_PATH + 'val_catboost_ready.csv', index=False)
test.to_csv(SAVE_PATH + 'test_catboost_ready.csv', index=False)

# Save CatBoost metadata
pd.DataFrame({'cat_feature': cat_features}).to_csv(
    SAVE_PATH + 'catboost_cat_features.csv', index=False
)

pd.DataFrame({
    'class_label': [0, 1],
    'class_weight': [class_weights[0], class_weights[1]]
}).to_csv(
    SAVE_PATH + 'catboost_class_weights.csv', index=False
)

pd.DataFrame({'feature': feature_cols}).to_csv(
    SAVE_PATH + 'catboost_feature_list.csv', index=False
)

print("\nSaved CatBoost prep files:")
print(" - train_catboost_ready.csv")
print(" - val_catboost_ready.csv")
print(" - test_catboost_ready.csv")
print(" - catboost_cat_features.csv")
print(" - catboost_class_weights.csv")
print(" - catboost_feature_list.csv")

gc.collect()

Loading Step 10 pruned files for CatBoost prep...
Feature cols     : 168
Categorical cols : 28
Numeric cols     : 140

Training class distribution:
Counter({np.int64(0): 455833, np.int64(1): 16599})
Class weights for CatBoost:
{0: 1.0, 1: 27.46147358274595}

Saved CatBoost prep files:
 - train_catboost_ready.csv
 - val_catboost_ready.csv
 - test_catboost_ready.csv
 - catboost_cat_features.csv
 - catboost_class_weights.csv
 - catboost_feature_list.csv


0

In [ ]:
# ============================================================
# STEP 11B: DNN Prep Pipeline
# Frequency encode categoricals
# Replace -999 sentinels from train statistics
# Scale all numeric inputs
# Apply SMOTE on training portion only
# Save DNN-ready files + preprocessing artifacts
# ============================================================

import gc
import os
import pickle
import pandas as pd
import numpy as np

from collections import Counter
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

print("Loading Step 10 pruned files for DNN prep...")

train_df = pd.read_csv(SAVE_PATH + 'train_step10_pruned.csv')
val_df   = pd.read_csv(SAVE_PATH + 'val_step10_pruned.csv')
test     = pd.read_csv(SAVE_PATH + 'test_step10_pruned.csv')

exclude_cols = ['isFraud']
if 'TransactionID' in train_df.columns:
    exclude_cols.append('TransactionID')

feature_cols = [c for c in train_df.columns if c not in exclude_cols]
cat_features = [c for c in feature_cols if train_df[c].dtype == 'object']
num_features = [c for c in feature_cols if c not in cat_features]

print(f"Feature cols     : {len(feature_cols)}")
print(f"Categorical cols : {len(cat_features)}")
print(f"Numeric cols     : {len(num_features)}")

# ------------------------------------------------------------
# 11B-1: Frequency encode categoricals using TRAIN ONLY
# ------------------------------------------------------------
print("\nFrequency encoding categorical columns...")

freq_maps = {}

for col in cat_features:
    freq_map = train_df[col].value_counts(dropna=False, normalize=True).to_dict()
    freq_maps[col] = freq_map

    train_df[col] = train_df[col].map(freq_map).fillna(0).astype(np.float32)
    val_df[col]   = val_df[col].map(freq_map).fillna(0).astype(np.float32)
    test[col]     = test[col].map(freq_map).fillna(0).astype(np.float32)

print(f"Frequency encoded {len(cat_features)} categorical columns.")

# Now all features are numeric
all_dnn_features = feature_cols.copy()

# ------------------------------------------------------------
# 11B-2: Replace -999 sentinels using TRAIN ONLY
# ------------------------------------------------------------
print("\nHandling -999 sentinels...")

sentinel_cols = [c for c in all_dnn_features if (train_df[c] == -999).any()]
sentinel_replacements = {}

for col in sentinel_cols:
    non_sentinel = train_df.loc[train_df[col] != -999, col]

    if len(non_sentinel) == 0:
        replacement = 0.0
    else:
        replacement = float(non_sentinel.median())

    sentinel_replacements[col] = replacement

    train_df[col] = train_df[col].replace(-999, replacement)
    val_df[col]   = val_df[col].replace(-999, replacement)
    test[col]     = test[col].replace(-999, replacement)

print(f"Replaced -999 in {len(sentinel_cols)} columns.")

# ------------------------------------------------------------
# 11B-3: StandardScaler on ALL DNN input features
# ------------------------------------------------------------
print("\nScaling DNN input features...")

scaler = StandardScaler()

train_df[all_dnn_features] = scaler.fit_transform(train_df[all_dnn_features]).astype(np.float32)
val_df[all_dnn_features]   = scaler.transform(val_df[all_dnn_features]).astype(np.float32)
test[all_dnn_features]     = scaler.transform(test[all_dnn_features]).astype(np.float32)

print("Scaling complete.")

# ------------------------------------------------------------
# 11B-4: SMOTE on TRAIN ONLY
# ------------------------------------------------------------
print("\nApplying SMOTE on DNN training set only...")

X_train = train_df[all_dnn_features].values.astype(np.float32)
y_train = train_df['isFraud'].values

print("Class distribution before SMOTE:", Counter(y_train))

smote = SMOTE(
    sampling_strategy=0.5,
    random_state=42,
    k_neighbors=5
)

X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE :", Counter(y_resampled))
print("Resampled shape                :", X_resampled.shape)

train_dnn = pd.DataFrame(X_resampled, columns=all_dnn_features)
train_dnn['isFraud'] = y_resampled

# Save validation/test already transformed but not oversampled
val_dnn = val_df.copy()
test_dnn = test.copy()

# ------------------------------------------------------------
# 11B-5: Save DNN outputs + artifacts
# ------------------------------------------------------------
train_dnn.to_csv(SAVE_PATH + 'train_dnn_ready.csv', index=False)
val_dnn.to_csv(SAVE_PATH + 'val_dnn_ready.csv', index=False)
test_dnn.to_csv(SAVE_PATH + 'test_dnn_ready.csv', index=False)

with open(SAVE_PATH + 'dnn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open(SAVE_PATH + 'dnn_freq_maps.pkl', 'wb') as f:
    pickle.dump(freq_maps, f)

with open(SAVE_PATH + 'dnn_sentinel_replacements.pkl', 'wb') as f:
    pickle.dump(sentinel_replacements, f)

pd.DataFrame({'feature': all_dnn_features}).to_csv(
    SAVE_PATH + 'dnn_feature_list.csv', index=False
)

print("\nSaved DNN prep files:")
print(" - train_dnn_ready.csv")
print(" - val_dnn_ready.csv")
print(" - test_dnn_ready.csv")
print(" - dnn_scaler.pkl")
print(" - dnn_freq_maps.pkl")
print(" - dnn_sentinel_replacements.pkl")
print(" - dnn_feature_list.csv")

gc.collect()

Loading Step 10 pruned files for DNN prep...
Feature cols     : 168
Categorical cols : 28
Numeric cols     : 140

Frequency encoding categorical columns...
Frequency encoded 28 categorical columns.

Handling -999 sentinels...
Replaced -999 in 51 columns.

Scaling DNN input features...
Scaling complete.

Applying SMOTE on DNN training set only...
Class distribution before SMOTE: Counter({np.int64(0): 455833, np.int64(1): 16599})
Class distribution after SMOTE : Counter({np.int64(0): 455833, np.int64(1): 227916})
Resampled shape                : (683749, 168)


In [ ]:
# ============================================================
# STEP 11C: Autoencoder Prep Pipeline
# Reuse DNN-style preprocessing
# NO SMOTE
# Train on normal rows only
# Save train-normal / val-eval / test-eval files
# ============================================================

import gc
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/feature_engineered/'

print("Loading Step 10 pruned files for Autoencoder prep...")

train_df = pd.read_csv(SAVE_PATH + 'train_step10_pruned.csv')
val_df   = pd.read_csv(SAVE_PATH + 'val_step10_pruned.csv')
test     = pd.read_csv(SAVE_PATH + 'test_step10_pruned.csv')

exclude_cols = ['isFraud']
if 'TransactionID' in train_df.columns:
    exclude_cols.append('TransactionID')

feature_cols = [c for c in train_df.columns if c not in exclude_cols]
cat_features = [c for c in feature_cols if train_df[c].dtype == 'object']

# ------------------------------------------------------------
# 11C-1: Frequency encode categoricals using TRAIN ONLY
# ------------------------------------------------------------
freq_maps_auto = {}

for col in cat_features:
    freq_map = train_df[col].value_counts(dropna=False, normalize=True).to_dict()
    freq_maps_auto[col] = freq_map

    train_df[col] = train_df[col].map(freq_map).fillna(0).astype(np.float32)
    val_df[col]   = val_df[col].map(freq_map).fillna(0).astype(np.float32)
    test[col]     = test[col].map(freq_map).fillna(0).astype(np.float32)

all_auto_features = feature_cols.copy()

# ------------------------------------------------------------
# 11C-2: Replace -999 sentinels using TRAIN ONLY
# ------------------------------------------------------------
sentinel_cols_auto = [c for c in all_auto_features if (train_df[c] == -999).any()]
sentinel_replacements_auto = {}

for col in sentinel_cols_auto:
    non_sentinel = train_df.loc[train_df[col] != -999, col]

    if len(non_sentinel) == 0:
        replacement = 0.0
    else:
        replacement = float(non_sentinel.median())

    sentinel_replacements_auto[col] = replacement

    train_df[col] = train_df[col].replace(-999, replacement)
    val_df[col]   = val_df[col].replace(-999, replacement)
    test[col]     = test[col].replace(-999, replacement)

# ------------------------------------------------------------
# 11C-3: Scale features using TRAIN ONLY
# ------------------------------------------------------------
scaler_auto = StandardScaler()

train_df[all_auto_features] = scaler_auto.fit_transform(train_df[all_auto_features]).astype(np.float32)
val_df[all_auto_features]   = scaler_auto.transform(val_df[all_auto_features]).astype(np.float32)
test[all_auto_features]     = scaler_auto.transform(test[all_auto_features]).astype(np.float32)

# ------------------------------------------------------------
# 11C-4: Keep ONLY normal training rows for autoencoder fit
# ------------------------------------------------------------
train_auto_normal = train_df.loc[train_df['isFraud'] == 0].copy()

# Validation/test remain full for evaluation later
val_auto_eval = val_df.copy()
test_auto_eval = test.copy()

print("Autoencoder training shape (normal only):", train_auto_normal.shape)
print("Validation eval shape                    :", val_auto_eval.shape)
print("Test eval shape                          :", test_auto_eval.shape)

# ------------------------------------------------------------
# 11C-5: Save outputs + artifacts
# ------------------------------------------------------------
train_auto_normal.to_csv(SAVE_PATH + 'train_autoencoder_normal.csv', index=False)
val_auto_eval.to_csv(SAVE_PATH + 'val_autoencoder_eval.csv', index=False)
test_auto_eval.to_csv(SAVE_PATH + 'test_autoencoder_eval.csv', index=False)

with open(SAVE_PATH + 'autoencoder_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_auto, f)

with open(SAVE_PATH + 'autoencoder_freq_maps.pkl', 'wb') as f:
    pickle.dump(freq_maps_auto, f)

with open(SAVE_PATH + 'autoencoder_sentinel_replacements.pkl', 'wb') as f:
    pickle.dump(sentinel_replacements_auto, f)

pd.DataFrame({'feature': all_auto_features}).to_csv(
    SAVE_PATH + 'autoencoder_feature_list.csv', index=False
)

print("\nSaved Autoencoder prep files:")
print(" - train_autoencoder_normal.csv")
print(" - val_autoencoder_eval.csv")
print(" - test_autoencoder_eval.csv")
print(" - autoencoder_scaler.pkl")
print(" - autoencoder_freq_maps.pkl")
print(" - autoencoder_sentinel_replacements.pkl")
print(" - autoencoder_feature_list.csv")

gc.collect()

#EDA for checking

In [ ]:
# ============================================================
# Table 4.4 Time-Based Train and Validation Split Summary
# ============================================================

split_summary = pd.DataFrame({
    "Dataset": ["Training Set", "Validation Set"],
    "Rows": [len(train_df), len(val_df)],
    "Columns": [train_df.shape[1], val_df.shape[1]],
    "Fraud Count": [
        int(train_df["isFraud"].sum()),
        int(val_df["isFraud"].sum())
    ],
    "Fraud Percentage": [
        round(train_df["isFraud"].mean() * 100, 4),
        round(val_df["isFraud"].mean() * 100, 4)
    ],
    "TransactionDT Min": [
        int(train_df["TransactionDT"].min()),
        int(val_df["TransactionDT"].min())
    ],
    "TransactionDT Max": [
        int(train_df["TransactionDT"].max()),
        int(val_df["TransactionDT"].max())
    ]
})

display(split_summary)

split_summary.to_csv(
    chapter4_dir + "table_4_4_time_based_split_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# Figure 4.7 Feature Count Reduction Across the Pipeline
# ============================================================

# Original V-column count from IEEE-CIS
v_before = 339

# Current V-column count after reduction
v_after = len([c for c in train_df.columns if c.startswith("V")])

# Overall feature counts
cleaned_feature_count = train.shape[1] - 1 if "train" in globals() else None
after_fe_feature_count = train_df.shape[1] - 1
catboost_count = catboost_feature_count if isinstance(catboost_feature_count, int) else np.nan
dnn_count = dnn_feature_count if isinstance(dnn_feature_count, int) else np.nan
ae_count = autoencoder_feature_count if isinstance(autoencoder_feature_count, int) else np.nan

feature_count_summary = pd.DataFrame({
    "Stage": [
        "Original V Features",
        "V Features After Reduction",
        "Cleaned Dataset Features",
        "After Feature Engineering",
        "CatBoost Features",
        "DNN Features",
        "Autoencoder Features"
    ],
    "Feature Count": [
        v_before,
        v_after,
        cleaned_feature_count,
        after_fe_feature_count,
        catboost_count,
        dnn_count,
        ae_count
    ]
})

display(feature_count_summary)

feature_count_summary.to_csv(
    chapter4_dir + "figure_4_7_feature_count_summary.csv",
    index=False
)

plt.figure(figsize=(9, 4))
plt.bar(feature_count_summary["Stage"], feature_count_summary["Feature Count"])
plt.title("Feature Count Reduction Across the Preparation Pipeline")
plt.xlabel("Pipeline Stage")
plt.ylabel("Number of Features")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(chapter4_dir + "figure_4_7_feature_count_reduction.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Figure 4.8 Fraud Rate by Email Risk Category
# ============================================================

if "P_email_risk" in train_df.columns:
    email_risk_summary = (
        train_df.groupby("P_email_risk")
        .agg(
            transaction_count=("isFraud", "count"),
            fraud_count=("isFraud", "sum"),
            fraud_rate=("isFraud", "mean")
        )
        .reset_index()
    )

    email_risk_summary["fraud_rate_percent"] = email_risk_summary["fraud_rate"] * 100

    display(email_risk_summary)

    email_risk_summary.to_csv(
        chapter4_dir + "figure_4_8_email_risk_summary.csv",
        index=False
    )

    plt.figure(figsize=(7, 4))
    plt.bar(
        email_risk_summary["P_email_risk"].astype(str),
        email_risk_summary["fraud_rate_percent"]
    )
    plt.title("Fraud Rate by Purchaser Email Risk Category")
    plt.xlabel("Purchaser Email Risk Category")
    plt.ylabel("Fraud Rate (%)")
    plt.tight_layout()
    plt.savefig(chapter4_dir + "figure_4_8_fraud_rate_by_email_risk.png", dpi=300)
    plt.show()

else:
    print("P_email_risk column not found. Check the email risk feature name.")

In [ ]:
# ============================================================
# Figure 4.9 Fraud Rate by UID Transaction Count Bucket
# ============================================================

if "UID1_txn_count" in train_df.columns:
    temp_df = train_df.copy()

    temp_df["UID1_txn_count_bucket"] = pd.cut(
        temp_df["UID1_txn_count"],
        bins=[-1, 0, 1, 3, 10, 50, np.inf],
        labels=[
            "First transaction",
            "1 previous txn",
            "2-3 previous txns",
            "4-10 previous txns",
            "11-50 previous txns",
            "More than 50 previous txns"
        ]
    )

    velocity_summary = (
        temp_df.groupby("UID1_txn_count_bucket", observed=True)
        .agg(
            transaction_count=("isFraud", "count"),
            fraud_count=("isFraud", "sum"),
            fraud_rate=("isFraud", "mean")
        )
        .reset_index()
    )

    velocity_summary["fraud_rate_percent"] = velocity_summary["fraud_rate"] * 100

    display(velocity_summary)

    velocity_summary.to_csv(
        chapter4_dir + "figure_4_9_velocity_summary.csv",
        index=False
    )

    plt.figure(figsize=(9, 4))
    plt.bar(
        velocity_summary["UID1_txn_count_bucket"].astype(str),
        velocity_summary["fraud_rate_percent"]
    )
    plt.title("Fraud Rate by UID1 Transaction Count Bucket")
    plt.xlabel("UID1 Transaction Count Bucket")
    plt.ylabel("Fraud Rate (%)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(chapter4_dir + "figure_4_9_fraud_rate_by_uid_velocity.png", dpi=300)
    plt.show()

else:
    print("UID1_txn_count column not found. Using night transaction flag instead.")

    if "is_night_txn" in train_df.columns:
        night_summary = (
            train_df.groupby("is_night_txn")
            .agg(
                transaction_count=("isFraud", "count"),
                fraud_count=("isFraud", "sum"),
                fraud_rate=("isFraud", "mean")
            )
            .reset_index()
        )

        night_summary["fraud_rate_percent"] = night_summary["fraud_rate"] * 100
        night_summary["Transaction Type"] = night_summary["is_night_txn"].map({
            0: "Non-night transaction",
            1: "Night transaction"
        })

        display(night_summary)

        night_summary.to_csv(
            chapter4_dir + "figure_4_9_night_transaction_summary.csv",
            index=False
        )

        plt.figure(figsize=(6, 4))
        plt.bar(
            night_summary["Transaction Type"],
            night_summary["fraud_rate_percent"]
        )
        plt.title("Fraud Rate by Night Transaction Flag")
        plt.xlabel("Transaction Type")
        plt.ylabel("Fraud Rate (%)")
        plt.tight_layout()
        plt.savefig(chapter4_dir + "figure_4_9_fraud_rate_by_night_transaction.png", dpi=300)
        plt.show()
    else:
        print("Neither UID1_txn_count nor is_night_txn was found.")